In [1]:
import numpy as np
import urllib.request
from metaphone import doublemetaphone
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss

In [2]:
def load_cmu_dictionary():

    url = "https://svn.code.sf.net/p/cmusphinx/code/trunk/cmudict/cmudict-0.7b"

    words = []
    phonemes = []
    word_to_phoneme = {}

    print("Loading CMU Dictionary...")

    response = urllib.request.urlopen(url)

    for line in response:

        line = line.decode("latin-1").strip()

        if line.startswith(";;;"):
            continue

        parts = line.split()

        if len(parts) < 2:
            continue

        word = parts[0].lower()
        
        if "(" in word:
            continue

        phoneme_sequence = " ".join(parts[1:])

        words.append(word)
        phonemes.append(phoneme_sequence)
        word_to_phoneme[word] = phoneme_sequence

    print(f"Total words loaded: {len(words)}")

    return words, phonemes, word_to_phoneme

In [3]:
def build_tfidf_vectors(phonemes):

    print("Building TF-IDF vectors...")

    vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=(2, 3)
    )

    vectors = vectorizer.fit_transform(phonemes)

    vectors = vectors.toarray().astype("float32")

    faiss.normalize_L2(vectors)

    print(f"Vector Shape: {vectors.shape}")

    return vectors, vectorizer

In [4]:
def build_faiss_index(vectors):

    print("Building FAISS Index...")

    dimension = vectors.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(vectors)

    print(f"Vectors Stored: {index.ntotal}")

    return index

In [5]:
words, phonemes, word_to_phoneme = load_cmu_dictionary()

vectors, vectorizer = build_tfidf_vectors(phonemes)

index = build_faiss_index(vectors)

print("\nSystem Ready!")

Loading CMU Dictionary...
Total words loaded: 125067
Building TF-IDF vectors...
Vector Shape: (125067, 647)
Building FAISS Index...
Vectors Stored: 125067

System Ready!


In [6]:
def search(query, top_k=10):

    query = query.lower().strip()

    print("\n" + "=" * 70)

    if query in word_to_phoneme:

        query_phoneme = word_to_phoneme[query]

        print(f"Query Word   : {query}")
        print("In CMU Dict  : Yes")
        print(f"Phoneme Used : {query_phoneme}")

    else:

        primary, secondary = doublemetaphone(query)

        query_phoneme = primary if primary else query

        print(f"Query Word   : {query}")
        print("In CMU Dict  : No (Using Double Metaphone)")
        print(f"Phoneme Used : {query_phoneme}")

    query_vector = vectorizer.transform(
        [query_phoneme]
    ).toarray().astype("float32")

    # Normalize query vector
    faiss.normalize_L2(query_vector)

    scores, indices = index.search(query_vector, top_k)

    print("\nTop Similar Words")
    print("-" * 90)

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):

        similarity = score * 100

        print(
            f"{rank+1:2}. "
            f"{words[idx]:20} | "
            f"{phonemes[idx]:35} | "
            f"Similarity: {similarity:.2f}%"
        )

In [9]:
search("night")


Query Word   : night
In CMU Dict  : Yes
Phoneme Used : N AY1 T

Top Similar Words
------------------------------------------------------------------------------------------
 1. nite                 | N AY1 T                             | Similarity: 100.00%
 2. night                | N AY1 T                             | Similarity: 100.00%
 3. knight               | N AY1 T                             | Similarity: 100.00%
 4. nye                  | N AY1                               | Similarity: 85.71%
 5. nigh                 | N AY1                               | Similarity: 85.71%
 6. nights'              | N AY1 T S                           | Similarity: 85.31%
 7. nights               | N AY1 T S                           | Similarity: 85.31%
 8. night's              | N AY1 T S                           | Similarity: 85.31%
 9. knights              | N AY1 T S                           | Similarity: 85.31%
10. knight's             | N AY1 T S                           | Si

In [10]:
search("header")


Query Word   : header
In CMU Dict  : Yes
Phoneme Used : HH EH1 D ER0

Top Similar Words
------------------------------------------------------------------------------------------
 1. header               | HH EH1 D ER0                        | Similarity: 100.00%
 2. headers              | HH EH1 D ER0 Z                      | Similarity: 92.17%
 3. heldor               | HH EH1 L D ER0                      | Similarity: 84.07%
 4. helder               | HH EH1 L D ER0                      | Similarity: 84.07%
 5. herder               | HH EH1 R D ER0                      | Similarity: 83.10%
 6. cheddar              | CH EH1 D ER0                        | Similarity: 82.84%
 7. eder                 | EH1 D ER0                           | Similarity: 81.84%
 8. deader               | D EH1 D ER0                         | Similarity: 80.93%
 9. tedder               | T EH1 D ER0                         | Similarity: 79.43%
10. haider               | HH EY1 D ER0                        

In [11]:
print(f"Total words loaded: {len(words)}")
print(f"Vector Shape: {vectors.shape}")
print(f"Vectors Stored: {index.ntotal}")

Total words loaded: 125067
Vector Shape: (125067, 647)
Vectors Stored: 125067


In [12]:
search("phone")


Query Word   : phone
In CMU Dict  : Yes
Phoneme Used : F OW1 N

Top Similar Words
------------------------------------------------------------------------------------------
 1. phone                | F OW1 N                             | Similarity: 100.00%
 2. fone                 | F OW1 N                             | Similarity: 100.00%
 3. foe                  | F OW1                               | Similarity: 91.47%
 4. faux                 | F OW1                               | Similarity: 91.47%
 5. pfohl                | F OW1 L                             | Similarity: 90.44%
 6. fohl                 | F OW1 L                             | Similarity: 90.44%
 7. foale                | F OW1 L                             | Similarity: 90.44%
 8. foal                 | F OW1 L                             | Similarity: 90.44%
 9. folk                 | F OW1 K                             | Similarity: 88.74%
10. phoned               | F OW1 N D                           | Sim

In [13]:
search("smith")


Query Word   : smith
In CMU Dict  : Yes
Phoneme Used : S M IH1 TH

Top Similar Words
------------------------------------------------------------------------------------------
 1. smith                | S M IH1 TH                          | Similarity: 100.00%
 2. smit                 | S M IH1 T                           | Similarity: 86.00%
 3. smithey              | S M IH1 TH IY0                      | Similarity: 84.00%
 4. smithee              | S M IH1 TH IY1                      | Similarity: 83.40%
 5. myth                 | M IH1 TH                            | Similarity: 82.02%
 6. smiths               | S M IH1 TH S                        | Similarity: 81.19%
 7. smith's              | S M IH1 TH S                        | Similarity: 81.19%
 8. smits                | S M IH1 T S                         | Similarity: 76.78%
 9. smear                | S M IH1 R                           | Similarity: 75.85%
10. smithson             | S M IH1 TH S AH0 N                  | S

In [14]:
search("history")


Query Word   : history
In CMU Dict  : Yes
Phoneme Used : HH IH1 S T ER0 IY0

Top Similar Words
------------------------------------------------------------------------------------------
 1. history              | HH IH1 S T ER0 IY0                  | Similarity: 100.00%
 2. history's            | HH IH1 S T ER0 IY0 Z                | Similarity: 94.18%
 3. histories            | HH IH1 S T ER0 IY0 Z                | Similarity: 94.18%
 4. hyster               | HH IH1 S T ER0                      | Similarity: 85.48%
 5. mystery              | M IH1 S T ER0 IY0                   | Similarity: 84.59%
 6. mysteries            | M IH1 S T ER0 IY0 Z                 | Similarity: 79.32%
 7. jittery              | JH IH1 T ER0 IY0                    | Similarity: 77.49%
 8. hillery              | HH IH1 L ER0 IY0                    | Similarity: 77.14%
 9. hilleary             | HH IH1 L ER0 IY0                    | Similarity: 77.14%
10. hillary              | HH IH1 L ER0 IY0             

In [15]:
search("katherine")


Query Word   : katherine
In CMU Dict  : Yes
Phoneme Used : K AE1 TH ER0 IH0 N

Top Similar Words
------------------------------------------------------------------------------------------
 1. katherine            | K AE1 TH ER0 IH0 N                  | Similarity: 100.00%
 2. catherine            | K AE1 TH ER0 AH0 N                  | Similarity: 86.21%
 3. matherne             | M AE1 TH ER0 N                      | Similarity: 83.41%
 4. hathorne             | HH AE1 TH ER0 N                     | Similarity: 79.47%
 5. rathert              | R AE1 TH ER0 T                      | Similarity: 78.83%
 6. mathur               | M AE1 TH ER0                        | Similarity: 77.97%
 7. atherton             | AE1 TH ER0 T AH0 N                  | Similarity: 77.78%
 8. sathre               | S AE1 TH ER0                        | Similarity: 77.69%
 9. matherson            | M AE1 TH ER0 S AH0 N                | Similarity: 76.66%
10. fathers'             | F AE1 TH ER0 Z             

In [16]:
search("teyradds")


Query Word   : teyradds
In CMU Dict  : No (Using Double Metaphone)
Phoneme Used : TRTS

Top Similar Words
------------------------------------------------------------------------------------------
 1. #pound-sign          | P AW1 N D S AY2 N                   | Similarity: 0.00%
 2. #hash-mark           | HH AE1 M AA2 R K                    | Similarity: 0.00%
 3. "unquote             | AH1 N K W OW1 T                     | Similarity: 0.00%
 4. "quote               | K W OW1 T                           | Similarity: 0.00%
 5. "in-quotes           | IH1 N K W OW1 T S                   | Similarity: 0.00%
 6. "end-quote           | EH1 N D K W OW1 T                   | Similarity: 0.00%
 7. "end-of-quote        | EH1 N D AH0 V K W OW1 T             | Similarity: 0.00%
 8. "double-quote        | D AH1 B AH0 L K W OW1 T             | Similarity: 0.00%
 9. "close-quote         | K L OW1 Z K W OW1 T                 | Similarity: 0.00%
10. !exclamation-point   | EH2 K S K L AH0 M EY1 SH AH0